# Walking skeleton — `bc_planning_optimizer.run`

End-to-end tracer for the planning-optimizer math seam — now driven by **real Item Ledger Entry data** from BC via the API Query extract (issue #12).

## The pipeline

1. `scripts/extract-ile-summary.sh` curls the AL API Query exposed by the `Bc Linux Smoke` app at `/api/fbakkensen/planningOptimizer/v1.0/itemLedgerSummaries`, paginates, and writes `.build/extracts/ile-summary.csv`.
2. `bc_planning_optimizer.run(extract_path)` reads the CSV, aggregates signed quantities to per-SKU daily demand (returns net automatically per [ADR 0006](../../docs/adr/0006-bootstrap-ltd-shared-monte-carlo-engine.md)), applies the placeholder lead-time constant, and writes `recommendations.json`.

## The math (deliberately naive)

```
daily_demand   = max(-mean(signed bucket quantities), 0)
reorder_point  = daily_demand × DEFAULT_LEAD_TIME_DAYS   # 7, placeholder
safety_stock   = reorder_point / 2
```

Real bootstrap LTD / SBA / AutoETS / simulator-verified policy lands in subsequent slices.

In [1]:
import json
import subprocess
from pathlib import Path

import pandas as pd

from bc_planning_optimizer import run

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = (NOTEBOOK_DIR / ".." / "..").resolve()
EXTRACT_PATH = REPO_ROOT / ".build" / "extracts" / "ile-summary.csv"
EXTRACT_PATH

PosixPath('/home/fbakkensen/repos/bc-on-linux-test/.build/extracts/ile-summary.csv')

## (Re)generate the extract

Requires the BC stack to be running (`cd bc-linux && docker compose up -d --wait`) and the `Bc Linux Smoke` app to be published.

Skip this cell if `.build/extracts/ile-summary.csv` is already fresh enough.

In [2]:
subprocess.run([str(REPO_ROOT / "scripts" / "extract-ile-summary.sh")], check=True)
EXTRACT_PATH.exists(), EXTRACT_PATH.stat().st_size

Fetching itemLedgerSummaries from http://localhost:7052/BC/api/fbakkensen/planningOptimizer/v1.0 for CRONUS International Ltd....
Writing /home/fbakkensen/repos/bc-on-linux-test/.build/extracts/ile-summary.csv (756 rows)...
✓ Extract landed at /home/fbakkensen/repos/bc-on-linux-test/.build/extracts/ile-summary.csv (756 data rows).


(True, 17473)

## A slice of the extract

Each row is one `(item_no, variant_code, location_code, posting_date)` bucket with the **signed** sum of Item Ledger Entry `Quantity`. Negatives are demand; positives are returns netting against it.

In [3]:
extract_df = pd.read_csv(
    EXTRACT_PATH,
    dtype={
        "item_no": "string",
        "variant_code": "string",
        "location_code": "string",
        "quantity": "float64",
    },
    parse_dates=["posting_date"],
    keep_default_na=False,
)
print(f"{len(extract_df)} rows, {extract_df['item_no'].nunique()} distinct items")
extract_df.head(8)

756 rows, 39 distinct items


,item_no,variant_code,location_code,posting_date,quantity
0,1896-S,,,2025-01-01,4.0
1,1896-S,,,2025-01-08,9.0
2,1896-S,,,2025-01-18,-5.0
3,1896-S,,,2025-01-24,-4.0
4,1896-S,,,2025-02-08,13.0
5,1896-S,,,2025-02-16,-7.0
6,1896-S,,,2025-02-20,-6.0
7,1896-S,,,2025-03-08,15.0


## Run the pipeline

`run(extract_path)` writes `recommendations.json` next to its input — here, next to the CSV in `.build/extracts/`.

In [4]:
output_path = run(EXTRACT_PATH)
output_path

PosixPath('/home/fbakkensen/repos/bc-on-linux-test/.build/extracts/recommendations.json')

## Recommendations on real BC data

One row per SKU triplet. Columns are the placeholder policy fields; future slices add `model_run_id`, `recommendation_grain`, confidence, etc.

In [5]:
payload = json.loads(output_path.read_text())
recs_df = pd.DataFrame(payload["recommendations"])
print(f"{len(recs_df)} recommendations")
recs_df.head(10)

43 recommendations


,item_no,variant_code,location_code,reorder_point,safety_stock
0,1896-S,,,6.250000,3.125000
1,1900-S,,,3.937500,1.968750
2,1906-S,,,3.400000,1.700000
3,1908-S,,,4.351351,2.175676
4,1920-S,,,5.981818,2.990909
5,1928-S,,,1.540000,0.770000
6,1936-S,,,0.000000,0.000000
7,1953-W,,,71.166667,35.583333
8,1960-S,,,2.476923,1.238462
9,1964-S,,,5.710526,2.855263
